# 07 — Spell Comparison

Compare spells across circles, elements, and schools.

In [ ]:
import logging
from pathlib import Path

from omega.config.spells import Spell
from omega.model.constants import (
    SKILLID_EVALINT, SKILLID_MAGERY, SKILLID_MAGICRESISTANCE, SKILLID_MEDITATION,
)
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, SpellScenario, run_spell_scenario,
)
from omega.reporting.tables import comparison_table, format_table_html
from omega.reporting.plots import spell_comparison
from omega.logging import setup_logging
from IPython.display import HTML

setup_logging(level=logging.ERROR)

SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)

MAGE = CombatantSpec(
    name="Mage",
    skills={SKILLID_MAGERY: 100, SKILLID_EVALINT: 100, SKILLID_MEDITATION: 100},
    str_=50, dex_=50, int_=120,
    class_levels={"IsMage": 5},
)
TARGET = CombatantSpec(
    name="Target", is_npc=True,
    str_=50, dex_=50, int_=50, hp=500,
    armor=ArmorSpec(ar=30),
)
ITERATIONS = 200
BASE_SEED = 42

def run_spell(spell_id, **kw):
    """Helper to run a single spell scenario."""
    return run_spell_scenario(
        SpellScenario(
            caster=kw.get("caster", MAGE), target=kw.get("target", TARGET),
            spell_id=spell_id, iterations=ITERATIONS,
            base_seed=BASE_SEED, npc_mode=kw.get("npc_mode", False),
        ),
        shard=shard,
    )

print("Setup complete.")

## 1. Circle Scaling

Standard spells from Circle 1 (Magic Arrow) to Circle 7 (Flame Strike).
Higher circles deal more damage but have higher mana cost and casting delay.

In [ ]:
circle_spells = {
    "C1 Magic Arrow": Spell.MAGIC_ARROW,
    "C2 Harm": Spell.HARM,
    "C3 Fireball": Spell.FIREBALL,
    "C4 Lightning": Spell.LIGHTNING,
    "C5 Mind Blast": Spell.MIND_BLAST,
    "C6 Energy Bolt": Spell.ENERGY_BOLT,
    "C7 Flame Strike": Spell.FLAME_STRIKE,
}

circle_results = {}
for label, spell_id in circle_spells.items():
    circle_results[label] = run_spell(spell_id)
    ds = circle_results[label].damage_stats
    r = circle_results[label].ratios
    print(f"  {label:20s}  mean={ds.mean:6.2f}  on_cast={circle_results[label].damage_stats_on_cast.mean:6.2f}  "
          f"fizzle={r.fizzle_rate:.0%}")

In [ ]:
spell_comparison(circle_results, title="Standard Spells: Circle Scaling")

In [ ]:
rows = comparison_table(
    circle_results,
    stats=["mean", "mean_on_cast", "median", "min", "max",
           "fizzle_rate", "resist_rate", "effective_dps"],
)
HTML(format_table_html(rows))

## 2. School Comparison

Compare damage spells from different schools at similar power levels:
Standard, Necromancy, Earth, and Holy.

In [ ]:
school_spells = {
    "Fireball (Standard)": Spell.FIREBALL,
    "Abyssal Flame (Necro)": Spell.ABYSSAL_FLAME,
    "Ice Strike (Earth)": Spell.ICE_STRIKE,
    "Divine Fury (Holy)": Spell.DIVINE_FURY,
}

school_results = {}
for label, spell_id in school_spells.items():
    school_results[label] = run_spell(spell_id)
    ds = school_results[label].damage_stats
    r = school_results[label].ratios
    print(f"  {label:28s}  mean={ds.mean:6.2f}  fizzle={r.fizzle_rate:.0%}")

In [ ]:
spell_comparison(school_results, title="School Comparison — Similar Circle Spells")

In [ ]:
rows = comparison_table(
    school_results,
    stats=["mean", "mean_on_cast", "median", "p5", "p95",
           "fizzle_rate", "resist_rate", "effective_dps",
           "elem_total_net"],
)
HTML(format_table_html(rows))

## 3. Single-Target vs AoE

AoE spells like Chain Lightning hit multiple targets. Per-target damage is
typically lower, but total damage across all targets can be significantly higher.

In [ ]:
# Single-target: Energy Bolt
energy_bolt = run_spell(Spell.ENERGY_BOLT)

# AoE: Chain Lightning against 3 targets
aoe_targets = [TARGET, TARGET, TARGET]
chain_cell = run_spell_scenario(
    SpellScenario(
        caster=MAGE, target=aoe_targets,
        spell_id=Spell.CHAIN_LIGHTNING,
        iterations=ITERATIONS, base_seed=BASE_SEED, npc_mode=False,
    ),
    shard=shard,
)

aoe_results = {"Energy Bolt (1 target)": energy_bolt, "Chain Lightning (3 targets)": chain_cell}
for label, cell in aoe_results.items():
    ds = cell.damage_stats
    print(f"  {label:30s}  mean={ds.mean:6.2f}  on_cast={cell.damage_stats_on_cast.mean:6.2f}")

In [ ]:
rows = comparison_table(
    aoe_results,
    stats=["mean", "mean_on_cast", "median", "min", "max",
           "fizzle_rate", "resist_rate", "effective_dps"],
)
HTML(format_table_html(rows))

## 4. NPC vs Player Mode

NPC mode bypasses `TryToCast` (no fizzle check, no mana cost) — simulating
how NPCs cast spells. Player mode goes through the full pipeline.

In [ ]:
player_mode = run_spell(Spell.FIREBALL, npc_mode=False)
npc_mode = run_spell(Spell.FIREBALL, npc_mode=True)

mode_results = {"Player Mode": player_mode, "NPC Mode": npc_mode}
for label, cell in mode_results.items():
    ds = cell.damage_stats
    r = cell.ratios
    print(f"  {label:15s}  mean={ds.mean:6.2f}  fizzle={r.fizzle_rate:.0%}  resist={r.resist_rate:.0%}")

In [ ]:
rows = comparison_table(
    mode_results,
    stats=["mean", "mean_on_cast", "median", "min", "max",
           "fizzle_rate", "resist_rate", "cast_rate", "effective_dps"],
)
HTML(format_table_html(rows))